**Mines Paris 2025**

Deep Learning for Image Analisis

Practical Session: Oct 21-2025

You should submit the notebook with the answers by email before the deadline.

Deadline: Oct 27-2025

etienne.decenciere@minesparis.psl.eu

Teaching assistants:

leo.dechaumet@minesparis.psl.eu

mihaela.dimitrova@minesparis.psl.eu

# Segmentation

In this notebook you will train and test a deep learning model to perform binary image segmentation. Segmentation is a very common tool in image analysis. In particular, binary segmentation is used to locate some particular elements in an image, which will be labeled in white while the rest of the image remains black, like in [this example](https://i.stack.imgur.com/i24FE.png):

<a href="https://i.sstatic.net/i24FE.png"><img src="https://i.sstatic.net/i24FE.png" title="segmentation"/></div>

This notebook deals with Whole Slide Images (WSI). Whole slide imaging, also known as virtual microscopy, refers to scanning a complete microscope slide and creating a single high-resolution digital file. For more information, see [this introduction](https://www.mbfbioscience.com/whole-slide-imaging-analysis).

In [ ]:
import os
import json
from datetime import datetime
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import imageio.v2 as imageio
import numpy as np
from scipy import ndimage
import pandas as pd
import matplotlib.pyplot as plt
import random

## Data

Drag and drop the tissue-20221017T082544Z-001.zip into the colab files. Once done run the cell below to unzip the files into the 'data' folder.

Have a look at the different images in the folder, both initial images and their associated binary mask. As you can see some of them are quite messy with dirt, pen marks... The objective of this work is to create a model to segment these WSI images to remove these unwanted details.

In [ ]:
!unzip tissue-20221017T082544Z-001.zip -d data

# Fetching data

First step is to implement functions to send data to the training and testing pipeline. To do so, we implement the class `DataGeneratorTissue` used to select patches of images by customising PyTorch's `Dataset` class.

The function `get_random_patch_coord` randomly selects the coordinates of an image's patch, or crop. A `patch_coord` point to the top left corner of the corresponding patch. Thus the coordinates [y, x] gives for a patch of size [h, w] the image's patch *image* [y:y+h, x:x+w].


The function `choose_patch_coord` calls `get_random_patch_coord` and should return the coordinates of a patch with at most 70% of background labels (pixel value of 0 in label). If it cannot be achieved after 10 tryouts, the function should return the coordinates of the patch with the lowest background labels found.

The original Ground Truth label in the dataset is a binary image (single channel).

In this notebook, we want to use 2-channel label - the first channel represents the background, while the second one is the foreground, containing the objects of interest (the original binary label image).

Note: unlike Keras' `Sequence`, a PyTorch `Dataset` only needs to return **one** sample per call to `__getitem__` (here one randomly-cropped patch). Batching and shuffling across samples is handled separately by a `DataLoader`.

In [ ]:
class DataGeneratorTissue(Dataset):

    def __init__(self, path, prefix,
                 patch_size, nb_channels_in=3, nb_channels_out=2,
                 geometric_augmentations=[], augmentation_ratio=None):

        self.path = path
        self.prefix = prefix
        self.get_data_dict()
        self.patch_size = patch_size
        self.nb_channels_in = nb_channels_in
        self.nb_channels_out = nb_channels_out
        self.geometric_augmentations = geometric_augmentations
        self.augmentation_ratio = augmentation_ratio

    def __len__(self):
        return len(self.data)

    def __getitem__(self, data_idx):
        x, y = self.fetch_data(data_idx)
        return x, y

    def get_data_dict(self):
        self.data = {}
        for data_idx, image_name in enumerate(os.listdir(os.path.join(*[self.path, self.prefix, "jpg"]))):
            self.data[data_idx] = image_name

    def get_random_patch_coord(self, image_size):
        '''
        image_size: (height, width)
        '''

        ### TO BE IMPLEMENTED ### (hint: use np.random.randint)

        #########################

        return patch_coord

    def choose_patch_coord(self, image_shape, label, threshold=0.7):
        '''
        image_shape: (height, width)
        label: shape (height, width, channels)
        '''

        ### TO BE IMPLEMENTED ###

        #########################

        return best_patch_coord

    def fetch_data(self, data_idx):
        image_file_path = os.path.join(*[self.path, self.prefix, "jpg", self.data[data_idx]])
        image = imageio.imread(image_file_path)

        label_file_path = os.path.join(*[self.path, self.prefix, "lbl", self.data[data_idx]])
        label = imageio.imread(label_file_path)

        image = np.float32(image/255)
        label = np.expand_dims(np.float32(label/255), axis=-1)

        patch_coord = self.choose_patch_coord(list(image.shape[:2]), label)

        image_patch = image[patch_coord[0]:patch_coord[0]+self.patch_size[0],
                            patch_coord[1]:patch_coord[1]+self.patch_size[1],:]

        label_patch = np.empty([*self.patch_size, 2], dtype=np.float32)
        original_label = label[patch_coord[0]:patch_coord[0]+self.patch_size[0],
                            patch_coord[1]:patch_coord[1]+self.patch_size[1], 0]


        ### TO BE IMPLEMENTED ###
        label_patch[:, :, 0] =                 # background
        label_patch[:, :, 1] =                 # foreground

        #########################

        if len(self.geometric_augmentations) != 0:
            if np.random.binomial(1, self.augmentation_ratio) == 1:
                augm_idx = random.randint(0,len(self.geometric_augmentations)-1)
                image_patch, label_patch = self.geometric_augmentations[augm_idx](image_patch, label_patch)

        # PyTorch expects channels-first tensors: (C, H, W)
        image_patch = torch.from_numpy(np.ascontiguousarray(image_patch.transpose(2, 0, 1))).float()
        label_patch = torch.from_numpy(np.ascontiguousarray(label_patch.transpose(2, 0, 1))).float()

        return image_patch, label_patch

The following code tests whether your DataGeneratorTissue class works properly. Try running it several times to check that all patches are different, have the correct dimensions, and that the labels are correctly generated.

In [ ]:
path = "data/tissue"
patch_size = [128, 128]
nb_channels_out = 2
val_dataset = DataGeneratorTissue(path, "val",
                                  patch_size, 3, nb_channels_out)

image_patch, label_patch = val_dataset[0]

plt.title("input image")
plt.imshow(image_patch.permute(1, 2, 0))
plt.show()
plt.title("background")
plt.imshow(label_patch[0], cmap="gray")
plt.show()
plt.title("tissue")
plt.imshow(label_patch[1], cmap="gray")
plt.show()

# UNet model builder

The deep learning model we will use is a U-Net model, which was initially introduced to perform segmentation on microscopy images (see [paper](https://arxiv.org/abs/1505.04597)).

Implement the encoder and decoder parts of the UNet model with skip connections in the `UNetModel` class using only `conv_block`, `nn.MaxPool2d`, `up_sampler` and [`torch.cat`](https://pytorch.org/docs/stable/generated/torch.cat.html) functions.

In PyTorch, a model is defined as a class inheriting from `nn.Module`: the layers used by the model are declared in `__init__`, and the way an input tensor flows through them is described in `forward`. You need to fill in **both** parts: the layers to declare in `__init__`, and the corresponding computations (including skip connections) in `forward`.

In [ ]:
class VerySimpleModel(nn.Module):

    def __init__(self, nb_channels_in, nb_channels_out):
        super().__init__()
        self.conv1 = nn.Conv2d(nb_channels_in, 256, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(256, 1, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        outputs = torch.sigmoid(x)
        return outputs

def build_very_simple_model(nb_channels_in, nb_channels_out):
    return VerySimpleModel(nb_channels_in, nb_channels_out)

In [ ]:
def conv_block(in_channels, filters, stride):
    conv_block = nn.Sequential(nn.Conv2d(in_channels, filters, kernel_size=3,
                                          stride=1, padding=1),
                                nn.BatchNorm2d(filters),
                                nn.LeakyReLU(),
                                nn.Conv2d(filters, filters, kernel_size=3,
                                          stride=stride, padding=1),
                                nn.BatchNorm2d(filters),
                                nn.LeakyReLU())
    return conv_block

def final_block(in_channels, filters, last_activation):
    if last_activation == "softmax":
        activation = nn.Softmax(dim=1)
    elif last_activation == "sigmoid":
        activation = nn.Sigmoid()
    else:
        activation = nn.Identity()
    conv_block = nn.Sequential(nn.Conv2d(in_channels, filters, kernel_size=3,
                                          stride=1, padding=1),
                                nn.BatchNorm2d(filters),
                                activation)
    return conv_block

def up_sampler(in_channels, filters):
    up_sampler = nn.Sequential(nn.ConvTranspose2d(in_channels, filters, kernel_size=2,
                                                    stride=2, padding=0),
                                nn.BatchNorm2d(filters),
                                nn.LeakyReLU())
    return up_sampler

class UNetModel(nn.Module):

    def __init__(self, nb_channels_in, nb_channels_out,
                 unet_filters, last_activation):
        super().__init__()

        ### TO BE IMPLEMENTED ###
        # Encoder part
        # (build the successive conv_block's, e.g. store them in a
        # nn.ModuleList self.encoder_blocks; downsampling is done by
        # using stride=2 in the last conv of each conv_block, following
        # the unet_filters list)

        # Decoder part
        # (build the successive up_sampler's and conv_block's needed to
        # go back up, e.g. self.upsamplers and self.decoder_blocks;
        # remember each decoder conv_block takes as input the
        # concatenation of the upsampled features and the matching
        # skip connection)

        #########################

        self.final_block = final_block(unet_filters[0], nb_channels_out, last_activation)

    def forward(self, x):
        skip = []

        ### TO BE IMPLEMENTED ###
        # Encoder part
        # (apply self.encoder_blocks to x, appending the feature map
        # produced before each downsampling step to the skip list)

        # Decoder part
        # (apply self.upsamplers to go back up, concatenating with the
        # matching skip connection along the channel dimension with
        # torch.cat, then apply self.decoder_blocks)

        #########################

        outputs = self.final_block(x)
        return outputs

def build_unet_model(nb_channels_in, nb_channels_out,
                     unet_filters, last_activation):
    unet_model = UNetModel(nb_channels_in, nb_channels_out, unet_filters, last_activation)
    return unet_model

# Segmentation loss

The loss used here is the Jaccard loss, also known as Intersection over Union. As we are working with binary images as outputs, we want to maximize the overlap area of ground truth and predicted masks (see [Wikipedia](https://fr.wikipedia.org/wiki/Indice_et_distance_de_Jaccard) for more details).

Implement the computation of the cardinal of the intersection and the union between *y_true* and *y_pred*, stored in the variables *intersection* and *union* for the jaccard index-based loss. The function *torch.sum()* should be sufficient to do so.

In [ ]:
def jaccard_loss(y_true, y_pred, smooth=1.0):
    y_true_f = y_true.reshape(-1)
    y_pred_f = y_pred.reshape(-1)

    ### TO BE IMPLEMENTED ###

    #########################

    return 1 - (intersection + smooth) / (union + smooth)

# Data augmentation

Data augmentation enables to generate new elements for our data set by applying some basic transformations to our images. For more examples, see [this tutorial](https://machinelearningmastery.com/how-to-configure-image-data-augmentation-when-training-deep-learning-neural-networks/).

In [ ]:
def flip_aug(image, label):
    flipped_image = np.flip(image, axis=1)
    flipped_label = np.flip(label, axis=1)
    return flipped_image, flipped_label

def rot_aug(image, label):
    angle = random.randint(-20, 20)
    image_rotation = ndimage.rotate(image, angle=angle, reshape=False)
    label_rotation = ndimage.rotate(label, angle=angle, reshape=False)
    return image_rotation, label_rotation

# Model's training

Launch the training of a UNet with the given hyperparameters, on the GPU if one is available.
A directory named "models", where are located the models' directories storing the data of the differents models trained, will be created.
In a model's directory, named according the model's id and the timestamp of its training (a security to avoid accidental overwriting), are stored the *model_parameters.json*, *best_model_weights.pt* and *log.csv* files.
Nothing to implement, just run for training.

In [ ]:
path = "data/tissue"

models_dir = "models"
if os.path.exists(models_dir) == False:
    os.mkdir(models_dir)

model_id = 1

dt = datetime.now()
timestamp = str(dt.hour) + ':' + str(dt.minute) + ':' + str(dt.second) + '-' + str(dt.day) + ':' + str(dt.month) + ':' + str(dt.year)
model_name = "modelID=" + str(model_id) +  "_timestamp=" + timestamp

model_dir = os.path.join(models_dir, model_name)
if os.path.exists(model_dir) == False:
    os.mkdir(model_dir)

nb_channels_in = 3
nb_channels_out = 2
last_activation = "softmax"
unet_filters = [8, 16, 32, 64]
patch_size = [128, 128]
batch_size = 5
lr = 0.001
nb_epochs = 50

model_parameters = {"nb_channels_in": nb_channels_in,
                    "nb_channels_out": nb_channels_out,
                    "last_activation": last_activation,
                    "unet_filters": unet_filters,
                    "patch_size": patch_size,
                    "batch_size": batch_size,
                    "lr": lr,
                    "nb_epochs": nb_epochs}

json.dump(model_parameters, open(os.path.join(model_dir, "model_parameters.json"), "w"))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = build_unet_model(nb_channels_in, nb_channels_out, unet_filters, last_activation).to(device)
print(model)

train_dataset = DataGeneratorTissue(path, "train",
                                    patch_size, 3, nb_channels_out,
                                    geometric_augmentations=[flip_aug, rot_aug], augmentation_ratio=0.2)

val_dataset = DataGeneratorTissue(path, "val",
                                  patch_size, 3, nb_channels_out)

train_datagen = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_datagen = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

opt = torch.optim.Adam(model.parameters(), lr=lr)

model_log_file_path = os.path.join(model_dir, "log.csv")
best_model_weights_file_path = os.path.join(model_dir, "best_model_weights.pt")

In [ ]:
best_val_loss = float("inf")
log_rows = []

for epoch in range(nb_epochs):

    model.train()
    train_losses = []
    for x, y in train_datagen:
        x, y = x.to(device), y.to(device)

        opt.zero_grad()
        y_pred = model(x)
        loss = jaccard_loss(y, y_pred)
        loss.backward()
        opt.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_datagen:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = jaccard_loss(y, y_pred)
            val_losses.append(loss.item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    log_rows.append({"epoch": epoch, "loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_weights_file_path)

    pd.DataFrame(log_rows).to_csv(model_log_file_path, index=False)

    print(f"epoch {epoch+1}/{nb_epochs} - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

# Model's training summary

Plot the training and validation loss.  
Nothing to implement.

In [ ]:
df = pd.read_csv(model_log_file_path, sep=",")
plt.plot(df["epoch"], df["loss"], "r", label="training loss")
plt.plot(df["epoch"], df["val_loss"], "b", label="validation loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend()

# Model's inference function

*make_apply_unet_model* is the function to call to embeds a UNet model into a function in charge of the whole pipeline to get from an 8-bit image an 8-bit binary segmentation mask.  
Implement the *apply_unet_model* function.
Here the pipeline should include:  
* a [0,1]-normalization  
* a padding of the image (so that its dimension are 2^{unet_depth = len(unet_filters)-1} divisible)  
* the application of the UNet model  
* an unpadding of the UNet model's output (to retrieve the initial image dimensions)  
* the conversion of the UNet model's output into a binary segmentation mask by taking the label with the highest probability for each pixel  

The *pad_image* and *unpad_image* are already provided.

In [ ]:
def pad_image(image, unet_depth):
    shape = image.shape[:2]
    if shape[0]%2**unet_depth != 0 or shape[1]%2**unet_depth != 0:
        new_shape = [shape[0] + 2**unet_depth - shape[0]%2**unet_depth,
                     shape[1] + 2**unet_depth - shape[1]%2**unet_depth]
        new_image = np.empty([*new_shape, 3])
        new_image[0:shape[0], 0:shape[1], :] = image[...]

        new_image[0:shape[0],shape[1]:new_shape[1],:] = image[:,shape[1]-(new_shape[1]-shape[1]):shape[1],:]
        new_image[shape[0]:new_shape[0],0:shape[1],:] = image[shape[0]-(new_shape[0]-shape[0]):shape[0],:,:]
        new_image[shape[0]:new_shape[0],shape[1]:new_shape[1],:] = image[shape[0]-(new_shape[0]-shape[0]):shape[0],shape[1]-(new_shape[1]-shape[1]):shape[1],:]

        return new_image
    else:
        return image

def unpad_image(image, old_shape):
    im_shape = image.shape[:2]
    if im_shape[0] == old_shape[0] and im_shape[1] == old_shape[1]:
        return image
    else:
        new_image = image[0:old_shape[0], 0:old_shape[1]]
        return new_image

def apply_unet_model(unet_model, unet_depth, image, device="cpu"):

    ### TO BE IMPLEMENTED ###
    # hints:
    # - normalize `image` to [0, 1]
    # - pad it with pad_image(image, unet_depth)
    # - convert it to a (1, C, H, W) float tensor on `device`
    # - run unet_model on it inside a torch.no_grad() block (put the
    #   model in eval() mode first)
    # - convert the output back to a (H, W, nb_channels_out) numpy array
    # - unpad it with unpad_image(..., image.shape[:2])
    # - take the argmax over channels to get the binary segmentation mask

    #########################
    return segm_binary

def make_apply_unet_model(unet_model, unet_depth, device="cpu"):
    return lambda image: apply_unet_model(unet_model, unet_depth, image, device)

# Model's application on test dataset

Apply a UNet model on test dataset images.  
The model is chosen with its id.

In [ ]:
indexes_to_test = [8, 42]  # indexes in [0, 42]

In [ ]:
device = torch.device("cpu")

model_idx = -1

models_list = sorted(os.listdir("models"))
model_dir = os.path.join("models", models_list[model_idx])

with open(os.path.join(model_dir, "model_parameters.json"), "r") as model_parameters_file:
    model_parameters = json.load(model_parameters_file)

weights_path = os.path.join(model_dir, "best_model_weights.pt")

model2 = build_unet_model(model_parameters["nb_channels_in"],
                          model_parameters["nb_channels_out"],
                          model_parameters["unet_filters"],
                          model_parameters["last_activation"]).to(device)

model2.load_state_dict(torch.load(weights_path, map_location=device))
model2.eval()

apply_model =  make_apply_unet_model(model2, len(model_parameters["unet_filters"])-1, device)

test_images_dir = os.path.join(path, "test")
test_images_filenames = os.listdir(os.path.join(test_images_dir, "jpg"))

for test_image_idx in indexes_to_test:

  image = imageio.imread(os.path.join(*[test_images_dir, "jpg", test_images_filenames[test_image_idx]]))
  label = imageio.imread(os.path.join(*[test_images_dir, "lbl", test_images_filenames[test_image_idx]]))

  segm = apply_model(image)

  fig, ax = plt.subplots(1, 3, figsize=(18, 18))

  ax[0].imshow(image)
  ax[0].set_title("Input")
  ax[1].imshow(segm, cmap="gray")
  ax[1].set_title("Prediction")
  ax[2].imshow(label, cmap="gray")
  ax[2].set_title("Target")

# Interpretation

Change values of indexes to test with different images.

What do you think about the results? What seems to work? What limitations do you see?